In [ ]:
#@title Imports

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import random
import plotly.express as px
import plotly.graph_objects as go
import os
import time
import networkx as nx
import community as community_louvain
import warnings
import json


from matplotlib.patches import Rectangle
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import pairwise_distances
from numba import jit, prange
from tqdm import tqdm
from collections import defaultdict
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.metrics import jaccard_score

warnings.filterwarnings('ignore')

In [ ]:
#@title Load DF

df_base = pd.read_csv(r"..\data\user_anime_top2000.csv")
df = df_base.set_axis(["titleID", "userID", "review_score", "favorited"], axis=1)

# Pivot tables
review_matrix = df.pivot_table(index="titleID", columns="userID", values="review_score")
favorite_matrix = df.pivot_table(index="titleID", columns="userID", values="favorited")

# Fill for cosine (zeros for unrated)
review_matrix_cosine = review_matrix.fillna(0)

# Fill for Jaccard (binary: rated = 1, not rated = 0)
review_binary = review_matrix.notna().astype(int)
favorite_binary = favorite_matrix.fillna(0).astype(int)

df.head()

In [ ]:
# Define file paths for saving
PIVOT_DIR = r"..\data\pivot_tables_filtered_top2000"

# Create directory if it doesn't exist
os.makedirs(PIVOT_DIR, exist_ok=True)

def save_pivot_tables(review_matrix, favorite_matrix, review_matrix_cosine, review_binary, favorite_binary):
    """Save all pivot tables to CSV files"""
    print("Saving pivot tables...")
    
    # Save the main matrices
    review_matrix.to_csv(os.path.join(PIVOT_DIR, "review_matrix.csv"))
    print("✓ Saved review_matrix.csv")
    
    favorite_matrix.to_csv(os.path.join(PIVOT_DIR, "favorite_matrix.csv"))
    print("✓ Saved favorite_matrix.csv")
    
    review_matrix_cosine.to_csv(os.path.join(PIVOT_DIR, "review_matrix_cosine.csv"))
    print("✓ Saved review_matrix_cosine.csv")
    
    review_binary.to_csv(os.path.join(PIVOT_DIR, "review_binary.csv"))
    print("✓ Saved review_binary.csv")
    
    favorite_binary.to_csv(os.path.join(PIVOT_DIR, "favorite_binary.csv"))
    print("✓ Saved favorite_binary.csv")
    
    # Save metadata
    metadata = {
        'review_matrix_shape': review_matrix.shape,
        'favorite_matrix_shape': favorite_matrix.shape,
        'review_sparsity': 1 - (review_matrix_cosine.values != 0).sum() / review_matrix_cosine.size,
        'favorite_sparsity': 1 - (favorite_binary.values != 0).sum() / favorite_binary.size,
    }
    
    pd.Series(metadata).to_csv(os.path.join(PIVOT_DIR, "metadata.csv"))
    print("✓ Saved metadata.csv")
    
    print(f"\nAll pivot tables saved to: {PIVOT_DIR}")

def load_pivot_tables():
    """Load pivot tables from CSV files"""
    print("Loading pivot tables...")
    
    try:
        review_matrix = pd.read_csv(os.path.join(PIVOT_DIR, "review_matrix.csv"), index_col=0)
        print("✓ Loaded review_matrix.csv")
        
        favorite_matrix = pd.read_csv(os.path.join(PIVOT_DIR, "favorite_matrix.csv"), index_col=0)
        print("✓ Loaded favorite_matrix.csv")
        
        review_matrix_cosine = pd.read_csv(os.path.join(PIVOT_DIR, "review_matrix_cosine.csv"), index_col=0)
        print("✓ Loaded review_matrix_cosine.csv")
        
        review_binary = pd.read_csv(os.path.join(PIVOT_DIR, "review_binary.csv"), index_col=0)
        print("✓ Loaded review_binary.csv")
        
        favorite_binary = pd.read_csv(os.path.join(PIVOT_DIR, "favorite_binary.csv"), index_col=0)
        print("✓ Loaded favorite_binary.csv")
        
        # Load and display metadata
        metadata = pd.read_csv(os.path.join(PIVOT_DIR, "metadata.csv"), index_col=0).iloc[:, 0]
        print("✓ Loaded metadata.csv")
        
        print(f"\nMatrix dimensions:")
        print(f"Review matrix: {review_matrix.shape}")
        print(f"Favorite matrix: {favorite_matrix.shape}")
        print(f"Review sparsity: {metadata['review_sparsity']:.2%}")
        print(f"Favorite sparsity: {metadata['favorite_sparsity']:.2%}")
        
        return review_matrix, favorite_matrix, review_matrix_cosine, review_binary, favorite_binary
        
    except FileNotFoundError as e:
        print(f"Error: Could not find pivot table files in {PIVOT_DIR}")
        print("Make sure to run the normalization and pivot table creation first!")
        return None

def check_pivot_tables_exist():
    """Check if pivot tables already exist"""
    required_files = [
        "review_matrix.csv",
        "favorite_matrix.csv", 
        "review_matrix_cosine.csv",
        "review_binary.csv",
        "favorite_binary.csv"
    ]
    
    existing_files = []
    missing_files = []
    
    for file in required_files:
        filepath = os.path.join(PIVOT_DIR, file)
        if os.path.exists(filepath):
            existing_files.append(file)
        else:
            missing_files.append(file)
    
    return existing_files, missing_files

# Main execution logic
print("=" * 60)
print("PIVOT TABLE CHECKPOINT SYSTEM")
print("=" * 60)

existing_files, missing_files = check_pivot_tables_exist()

if len(existing_files) == 5:
    print("✓ All pivot tables found! Loading from saved files...")
    result = load_pivot_tables()
    
    if result:
        review_matrix, favorite_matrix, review_matrix_cosine, review_binary, favorite_binary = result
        print("\n✓ All pivot tables loaded successfully!")
        print("You can now proceed directly to similarity calculations.")
    
elif len(existing_files) > 0:
    print(f"⚠ Partial pivot tables found ({len(existing_files)}/5 files)")
    print("Existing files:", existing_files)
    print("Missing files:", missing_files)
    print("\nRecommendation: Delete existing files and regenerate all pivot tables.")
    
    user_input = input("Do you want to regenerate all pivot tables? (y/n): ").lower()
    if user_input == 'y':
        # Clear existing files
        for file in existing_files:
            os.remove(os.path.join(PIVOT_DIR, file))
        print("Existing files cleared. Please run the normalization code.")
    else:
        print("Keeping existing files. Manual cleanup may be needed.")

else:
    print("No existing pivot tables found.")
    print("Please run the normalization and pivot table creation code first.")
    print("\nAfter creating the pivot tables, add this code to save them:")
    print()
    print("# Add this after creating your pivot tables:")
    print("save_pivot_tables(review_matrix, favorite_matrix, review_matrix_cosine, review_binary, favorite_binary)")

# Quick recovery script for similarity calculations
print("\n" + "=" * 60)
print("QUICK RECOVERY FOR SIMILARITY CALCULATIONS")
print("=" * 60)
print("If you need to restart similarity calculations, just run:")
print()
print("# Load pivot tables")
print("review_matrix, favorite_matrix, review_matrix_cosine, review_binary, favorite_binary = load_pivot_tables()")
print()
print("# Then run similarity calculations")
print("# (your similarity calculation code here)")

In [ ]:
def jaccard_scipy(binary_matrix, show_progress=True):
    """
    Use SciPy's optimized pdist function for Jaccard distance.
    Often faster than manual vectorization, especially for moderate sizes.
    """
    if isinstance(binary_matrix, pd.DataFrame):
        data = binary_matrix.values
        index = binary_matrix.index
        matrix_name = "titles"
    else:
        data = binary_matrix
        index = range(len(data))
        matrix_name = "items"
    
    n_items = data.shape[0]
    
    if show_progress:
        print(f"Computing Jaccard similarity for {n_items} {matrix_name} using SciPy...")
    
    # SciPy computes Jaccard distance, so we convert to similarity
    jaccard_distances = pdist(data, metric='jaccard')
    jaccard_similarities = 1 - jaccard_distances
    
    # Convert condensed distance matrix to square matrix
    jaccard_matrix = squareform(jaccard_similarities)
    
    # Set diagonal to 1 (self-similarity)
    np.fill_diagonal(jaccard_matrix, 1.0)
    
    if show_progress:
        print(f"✓ SciPy Jaccard similarity matrix ({n_items}×{n_items}) completed!")
    
    return pd.DataFrame(jaccard_matrix, index=index, columns=index)



print("=" * 60)
print("COMPUTING SIMILARITY MATRICES")
print("=" * 60)

# 1. Cosine similarity on normalized review scores
print("\n1. Computing Cosine Similarity on normalized review scores...")
start_time = time.time()
cosine_sim = cosine_similarity(review_matrix_cosine)
cosine_sim_df = pd.DataFrame(cosine_sim, index=review_matrix.index, columns=review_matrix.index)
cosine_time = time.time() - start_time
print(f"   ✓ Cosine similarity completed in {cosine_time:.2f} seconds")
print(f"   Matrix shape: {cosine_sim_df.shape}")

# 2. Jaccard similarity on favorites (binary) - with progress bar
print("\n2. Computing Jaccard Similarity on favorites...")
start_time = time.time()
jaccard_fav_df = jaccard_scipy(favorite_binary)
jaccard_fav_time = time.time() - start_time
print(f"   ✓ Favorite Jaccard similarity completed in {jaccard_fav_time:.2f} seconds")
print(f"   Matrix shape: {jaccard_fav_df.shape}")

# 3. Jaccard similarity on review presence (binary) - with progress bar
print("\n3. Computing Jaccard Similarity on review presence...")
start_time = time.time()
jaccard_review_df = jaccard_scipy(review_binary)
jaccard_review_time = time.time() - start_time
print(f"   ✓ Review Jaccard similarity completed in {jaccard_review_time:.2f} seconds")
print(f"   Matrix shape: {jaccard_review_df.shape}")

# Optional: Use chunked version for more detailed progress (uncomment if desired)
# print("\n2b. Alternative: Chunked Jaccard with detailed progress...")
# jaccard_fav_df_chunked = jaccard_matrix_chunked_progress(favorite_binary, chunk_size=50)

print("\n" + "=" * 60)
print("SIMILARITY CALCULATION SUMMARY")
print("=" * 60)
print(f"Cosine similarity (normalized scores): {cosine_time:.2f} seconds")
print(f"Jaccard similarity (favorites):        {jaccard_fav_time:.2f} seconds")
print(f"Jaccard similarity (reviews):          {jaccard_review_time:.2f} seconds")
print(f"Total computation time:                {cosine_time + jaccard_fav_time + jaccard_review_time:.2f} seconds")

# Display sample results
print("\n" + "=" * 60)
print("SAMPLE SIMILARITY RESULTS")
print("=" * 60)

print("\nCosine Similarity (first 5x5 titles):")
print(cosine_sim_df.iloc[:5, :5].round(3))

print("\nJaccard Similarity - Favorites (first 5x5 titles):")
print(jaccard_fav_df.iloc[:5, :5].round(3))

print("\nJaccard Similarity - Reviews (first 5x5 titles):")
print(jaccard_review_df.iloc[:5, :5].round(3))

# Basic statistics about the similarity matrices
print("\n" + "=" * 60)
print("SIMILARITY MATRIX STATISTICS")
print("=" * 60)

print("\nCosine Similarity Stats:")
cosine_values = cosine_sim_df.values
cosine_upper_tri = cosine_values[np.triu_indices_from(cosine_values, k=1)]  # Exclude diagonal
print(f"  Mean similarity: {cosine_upper_tri.mean():.3f}")
print(f"  Std similarity:  {cosine_upper_tri.std():.3f}")
print(f"  Min similarity:  {cosine_upper_tri.min():.3f}")
print(f"  Max similarity:  {cosine_upper_tri.max():.3f}")

print("\nJaccard Similarity (Favorites) Stats:")
jaccard_fav_values = jaccard_fav_df.values
jaccard_fav_upper_tri = jaccard_fav_values[np.triu_indices_from(jaccard_fav_values, k=1)]
print(f"  Mean similarity: {jaccard_fav_upper_tri.mean():.3f}")
print(f"  Std similarity:  {jaccard_fav_upper_tri.std():.3f}")
print(f"  Min similarity:  {jaccard_fav_upper_tri.min():.3f}")
print(f"  Max similarity:  {jaccard_fav_upper_tri.max():.3f}")

print("\nJaccard Similarity (Reviews) Stats:")
jaccard_review_values = jaccard_review_df.values
jaccard_review_upper_tri = jaccard_review_values[np.triu_indices_from(jaccard_review_values, k=1)]
print(f"  Mean similarity: {jaccard_review_upper_tri.mean():.3f}")
print(f"  Std similarity:  {jaccard_review_upper_tri.std():.3f}")
print(f"  Min similarity:  {jaccard_review_upper_tri.min():.3f}")
print(f"  Max similarity:  {jaccard_review_upper_tri.max():.3f}")

print("\n" + "=" * 60)
print("SIMILARITY MATRICES READY FOR ANALYSIS!")
print("=" * 60)
print("Available matrices:")
print("- cosine_sim_df: Title similarities based on normalized user ratings")
print("- jaccard_fav_df: Title similarities based on user favorites")  
print("- jaccard_review_df: Title similarities based on which users reviewed them")

In [ ]:
#@title Generate Heatmap
α = 1 # Weight for cosine similarity 0 (heavily jaccard weighted) to 1 (heavily cosine weighted)
β = 0 # Weight for review jaccard similarity
γ = 1 - α - β # Weight for favorite jaccard similarity

if (α + β > 1):
  raise ValueError("α + β must be less than or equal to 1")


combined_sim_df = (
    α * cosine_sim_df +
    β * jaccard_review_df+
    γ * jaccard_fav_df
)



# === Load anime title mapping ===
anime_df = pd.read_csv(r"C:\Users\osher\Desktop\stuf\schul\year 4\semester 2\needle in a data haystack\data\anime.csv")
id_to_title = dict(zip(anime_df["MAL_ID"], anime_df["Name"]))

# === Copy matrix to avoid modifying original ===
sim_matrix = combined_sim_df.copy()

# === Distance + clustering ===
distance_matrix = 1 - sim_matrix
np.fill_diagonal(distance_matrix.values, 0)

linkage_matrix = linkage(squareform(distance_matrix), method="average")
order = leaves_list(linkage_matrix)

ordered_sim = sim_matrix.iloc[order, :].iloc[:, order]

# === Map titleIDs to names for plot only ===
x_labels = [id_to_title.get(i, str(i)) for i in ordered_sim.columns]
y_labels = [id_to_title.get(i, str(i)) for i in ordered_sim.index]

# === Plot interactive heatmap ===
fig = px.imshow(
    ordered_sim.values,
    labels=dict(x="", y="", color="Similarity"),
    x=x_labels,
    y=y_labels,
    color_continuous_scale="plasma",
    zmin=0, zmax=1,
    aspect="equal"
)

# Clean ticks for large datasets
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

# Use full names in hover tooltips
fig.update_traces(
    hovertemplate="Anime X: %{x}<br>Anime Y: %{y}<br>Similarity: %{z:.2f}<extra></extra>"
)

fig.update_layout(
    title="Clustered Anime Similarity Matrix (Interactive)",
    width=800, height=800
)

fig.show()

In [ ]:
#@title Generate Distribution Comparison Graph

# Filter relevant data
all_scores = df['review_score']
favorited_scores = df[df['favorited'] == 1]['review_score']

# Bin scores from 1 to 10
score_bins = list(range(1, 11))

# Compute frequency counts
all_counts = all_scores.value_counts().reindex(score_bins, fill_value=0).sort_index()
fav_counts = favorited_scores.value_counts().reindex(score_bins, fill_value=0).sort_index()

# Create subplots: 2 rows, 1 column
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('All Reviews', 'Favorited Reviews'))

# Add All Reviews line plot
fig.add_trace(go.Scatter(
    x=score_bins,
    y=all_counts,
    mode='lines+markers',
    name='All Reviews',
    line=dict(color='blue')
), row=1, col=1)

# Add Favorited Reviews line plot
fig.add_trace(go.Scatter(
    x=score_bins,
    y=fav_counts,
    mode='lines+markers',
    name='Favorited Reviews',
    line=dict(color='orange')
), row=2, col=1)

# Layout settings
fig.update_layout(
    title='Distribution of Review Scores',
    font=dict(size=18),
    xaxis=dict(dtick=1),
    yaxis=dict(title='Count'),
    template='plotly_white',
    height=600
)

# Also update the subplot axes
fig.update_yaxes(
    title_text="Count", 
    gridwidth=2,  # Grid line thickness
    gridcolor='lightgray',  # Optional: grid color
    row=1, col=1
)
fig.update_yaxes(
    title_text="Count", 
    gridwidth=2, 
    gridcolor='lightgray',
    row=2, col=1
)
fig.update_xaxes(
    title_text="Review Score", 
    gridwidth=2, 
    row=2, col=1
)

fig.show()

In [ ]:
#@title Generate Popularity Bar Plot


MAX_TITLE_LENGTH = 20
top_n = 50


# 1. Calculate total reviews - convert score columns to numeric first
score_cols = [f'Score-{i}' for i in range(1, 11)]
# Convert score columns to numeric (handles strings and NaN values)
for col in score_cols:
    anime_df[col] = pd.to_numeric(anime_df[col], errors='coerce')
anime_df['total_reviews'] = anime_df[score_cols].sum(axis=1)

# 2. Filter out rows where both English and Japanese names are missing
anime_df_filtered = anime_df.dropna(subset=["English name", "Japanese name"], how="all")

# 3. Use Japanese name if English name is missing
anime_df_filtered["display_name"] = anime_df_filtered["English name"].fillna(anime_df_filtered["Japanese name"])

# 4. Sort and select top N
anime_df_sorted = anime_df_filtered.sort_values(by='total_reviews', ascending=False)
top_anime = anime_df_sorted.head(top_n).copy()

# 5. Filter out rows with missing scores to avoid "unknowns"
top_anime = top_anime.dropna(subset=['Score'])

# 6. Truncate long names
top_anime["Short_English_Name"] = top_anime["display_name"].apply(
    lambda x: x if len(x) <= MAX_TITLE_LENGTH else x[:MAX_TITLE_LENGTH].rstrip() + "..."
)
# If duplicates exist, make unique by appending a count
from collections import defaultdict
name_counts = defaultdict(int)
unique_names = []
for name in top_anime["Short_English_Name"]:
    name_counts[name] += 1
    if name_counts[name] > 1:
        unique_names.append(f"{name} ({name_counts[name]})")
    else:
        unique_names.append(name)
top_anime["Short_English_Name_Unique"] = unique_names

# Ensure Score is float
top_anime['Score'] = pd.to_numeric(top_anime['Score'], errors='coerce')

# Define custom continuous color scale as list of RGB hex strings:
custom_colorscale = [
    [0.0, '#FF0000'],    # saturated red at 0%
    [0.5, '#8B008B'],    # dark magenta in the middle
    [1.0, '#0000FF']     # saturated blue at 100%
]

fig = px.bar(
    top_anime,
    x="Short_English_Name_Unique",
    y="total_reviews",
    color=top_anime["Score"],  # continuous numeric scores
    color_continuous_scale=custom_colorscale,
    title=f"Top {top_n} Most Reviewed Anime (English/Japanese Titles)",
    labels={
        "Short_English_Name_Unique": "Anime",
        "total_reviews": "Number of Reviews",
        "color": "Avg. Score"
    },
    hover_data=["English name", "Japanese name", "Score"]
)

fig.update_layout(
    xaxis_tickangle=45,
    margin=dict(b=150),
    font=dict(size=15),
    height=800
)

fig.show()



In [ ]:
# Step 1: Get anime IDs from similarity matrix
used_ids = combined_sim_df.index.tolist()

# Step 2: Count reviews for each anime ID in your original review data
review_counts = df[df["titleID"].isin(used_ids)].groupby("titleID").size().reset_index(name="review_count")

# Step 3: Load metadata and merge
anime_df = pd.read_csv(r"C:\Users\osher\Desktop\stuf\schul\year 4\semester 2\needle in a data haystack\data\anime.csv")
anime_with_counts = anime_df.merge(review_counts, left_on="MAL_ID", right_on="titleID")

# Step 4: Sort by review count
sorted_anime = anime_with_counts.sort_values("review_count", ascending=False)

# Define title column width for clean alignment
max_title_len = 50

# Print header
print(f"{'Title':{max_title_len}} | {'MAL_ID':<8} | {'# Reviews':>10}")
print("-" * (max_title_len + 26))

# Print rows
for _, row in sorted_anime.iterrows():
    title = row['English name'] if (row['English name'] != "Unknown") else row['Name']
    print(f"{title[:max_title_len]:{max_title_len}} | {row['MAL_ID']:<8} | {row['review_count']:>10,}")

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Debug the community module import
print("Debugging community module import...")
try:
    import community
    print("✓ community module imported")
    print(f"Available attributes: {[attr for attr in dir(community) if not attr.startswith('_')]}")
    
    # Try to find the right function
    if hasattr(community, 'community_louvain'):
        import community.community_louvain as community_louvain
        print("✓ Using community.community_louvain")
    elif hasattr(community, 'best_partition'):
        community_louvain = community
        print("✓ Using community directly")
    else:
        print("✗ Cannot find best_partition function")
        print("\nTrying alternative imports...")
        
        # Try alternative import
        from community import community_louvain
        print("✓ Using from community import community_louvain")
except Exception as e:
    print(f"✗ Error importing community: {e}")
    print("\nPlease install with: pip install python-louvain")
    raise

print("\n" + "="*60 + "\n")

def analyze_resolution_impact(sim_matrix, threshold=0.3, resolutions=None):
    """
    Analyze how different resolution values affect community detection
    
    Higher resolution = smaller, more numerous communities
    Lower resolution = larger, fewer communities
    Default resolution = 1.0
    """
    
    if resolutions is None:
        # Test a range of resolutions
        resolutions = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0]
    
    # Create graph with specified threshold
    adj_matrix = sim_matrix.copy()
    adj_matrix[adj_matrix < threshold] = 0
    G = nx.from_pandas_adjacency(adj_matrix)
    G.remove_edges_from(nx.selfloop_edges(G))
    
    print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    print(f"Using threshold: {threshold}")
    print("\n" + "="*60)
    
    results = []
    
    for resolution in resolutions:
        print(f"\nTesting resolution = {resolution}")
        
        # Run Louvain with this resolution
        communities = community_louvain.best_partition(G, weight='weight', resolution=resolution)
        
        # Convert to DataFrame
        community_df = pd.DataFrame.from_dict(communities, orient='index', columns=['community'])
        community_df.index.name = 'anime'
        
        # Calculate statistics
        comm_sizes = community_df['community'].value_counts()
        n_communities = len(comm_sizes)
        largest_comm = comm_sizes.max()
        avg_size = comm_sizes.mean()
        median_size = comm_sizes.median()
        
        # Calculate modularity
        communities_sets = [set() for _ in range(max(communities.values()) + 1)]
        for node, comm_id in communities.items():
            communities_sets[comm_id].add(node)
        modularity = nx.algorithms.community.modularity(G, communities_sets, weight='weight')
        
        print(f"  Communities: {n_communities}")
        print(f"  Largest community: {largest_comm} anime")
        print(f"  Average size: {avg_size:.1f}")
        print(f"  Median size: {median_size:.1f}")
        print(f"  Modularity: {modularity:.4f}")
        
        # Size distribution
        size_ranges = {
            'tiny (1-5)': (comm_sizes <= 5).sum(),
            'small (6-20)': ((comm_sizes > 5) & (comm_sizes <= 20)).sum(),
            'medium (21-50)': ((comm_sizes > 20) & (comm_sizes <= 50)).sum(),
            'large (51-100)': ((comm_sizes > 50) & (comm_sizes <= 100)).sum(),
            'huge (>100)': (comm_sizes > 100).sum()
        }
        
        results.append({
            'resolution': resolution,
            'n_communities': n_communities,
            'largest_community': largest_comm,
            'avg_size': avg_size,
            'median_size': median_size,
            'modularity': modularity,
            'community_df': community_df,
            'size_distribution': size_ranges
        })
    
    return results

def visualize_resolution_results(results):
    """Create comprehensive visualization of resolution impact"""
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Impact of Resolution Parameter on Community Detection', fontsize=16)
    
    resolutions = [r['resolution'] for r in results]
    
    # 1. Number of communities vs resolution
    axes[0, 0].plot(resolutions, [r['n_communities'] for r in results], 'b-o')
    axes[0, 0].set_xlabel('Resolution')
    axes[0, 0].set_ylabel('Number of Communities')
    axes[0, 0].set_title('Community Count')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xscale('log')
    
    # 2. Largest community size vs resolution
    axes[0, 1].plot(resolutions, [r['largest_community'] for r in results], 'r-o')
    axes[0, 1].set_xlabel('Resolution')
    axes[0, 1].set_ylabel('Size of Largest Community')
    axes[0, 1].set_title('Largest Community Size')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xscale('log')
    
    # 3. Average community size vs resolution
    axes[0, 2].plot(resolutions, [r['avg_size'] for r in results], 'g-o', label='Mean')
    axes[0, 2].plot(resolutions, [r['median_size'] for r in results], 'g--o', label='Median')
    axes[0, 2].set_xlabel('Resolution')
    axes[0, 2].set_ylabel('Community Size')
    axes[0, 2].set_title('Average and Median Community Size')
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_xscale('log')
    axes[0, 2].legend()
    
    # 4. Modularity vs resolution
    axes[1, 0].plot(resolutions, [r['modularity'] for r in results], 'm-o')
    axes[1, 0].set_xlabel('Resolution')
    axes[1, 0].set_ylabel('Modularity')
    axes[1, 0].set_title('Modularity Score')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_xscale('log')
    
    # 5. Size distribution stacked bar chart
    size_categories = ['tiny (1-5)', 'small (6-20)', 'medium (21-50)', 'large (51-100)', 'huge (>100)']
    bottom = np.zeros(len(results))
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#ff99cc']
    
    for i, category in enumerate(size_categories):
        values = [r['size_distribution'][category] for r in results]
        axes[1, 1].bar(range(len(results)), values, bottom=bottom, label=category, color=colors[i])
        bottom += values
    
    axes[1, 1].set_xlabel('Resolution')
    axes[1, 1].set_ylabel('Number of Communities')
    axes[1, 1].set_title('Community Size Distribution')
    axes[1, 1].set_xticks(range(len(results)))
    axes[1, 1].set_xticklabels([f"{r:.1f}" for r in resolutions])
    axes[1, 1].legend()
    
    # 6. Efficiency plot (modularity vs largest community)
    scatter = axes[1, 2].scatter([r['largest_community'] for r in results], 
                                [r['modularity'] for r in results], 
                                c=resolutions, s=100, cmap='viridis')
    axes[1, 2].set_xlabel('Largest Community Size')
    axes[1, 2].set_ylabel('Modularity')
    axes[1, 2].set_title('Modularity vs Community Size Trade-off')
    axes[1, 2].grid(True, alpha=0.3)
    
    # Add resolution labels to points
    for i, res in enumerate(resolutions):
        axes[1, 2].annotate(f'{res}', 
                           ([r['largest_community'] for r in results][i], 
                            [r['modularity'] for r in results][i]),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.tight_layout()
    plt.show()

def find_optimal_resolution(results, max_community_size=50, min_modularity=0.3):
    """Find the best resolution based on your criteria"""
    
    print("\n" + "="*60)
    print("FINDING OPTIMAL RESOLUTION")
    print(f"Criteria: max community size <= {max_community_size}, modularity >= {min_modularity}")
    
    valid_results = []
    
    for r in results:
        if r['largest_community'] <= max_community_size and r['modularity'] >= min_modularity:
            valid_results.append(r)
            print(f"\n✓ Resolution {r['resolution']}: largest={r['largest_community']}, modularity={r['modularity']:.4f}")
    
    if not valid_results:
        print("\nNo resolution meets both criteria. Relaxing constraints...")
        # Find best compromise
        for r in results:
            if r['largest_community'] <= max_community_size * 1.5:  # Allow 50% larger
                valid_results.append(r)
    
    if valid_results:
        # Pick the one with highest modularity
        best = max(valid_results, key=lambda x: x['modularity'])
        print(f"\n🎯 RECOMMENDED: Resolution = {best['resolution']}")
        print(f"   - {best['n_communities']} communities")
        print(f"   - Largest: {best['largest_community']} anime")
        print(f"   - Modularity: {best['modularity']:.4f}")
        return best
    else:
        print("\nNo suitable resolution found. Try adjusting the threshold or criteria.")
        return None

# Main execution
print("=== LOUVAIN COMMUNITY DETECTION WITH RESOLUTION TUNING ===\n")

# Test different resolutions
results = analyze_resolution_impact(sim_matrix, threshold=0.3)

# Visualize the results
visualize_resolution_results(results)

# Find optimal resolution for your needs
optimal = find_optimal_resolution(results, max_community_size=50, min_modularity=0.3)

if optimal:
    # Use the optimal communities
    optimal_communities = optimal['community_df']
    
    # Save results
    optimal_communities.to_csv('anime_communities_optimal.csv')
    print(f"\nOptimal communities saved to 'anime_communities_optimal.csv'")
    
    # Show sample communities
    print("\n=== SAMPLE COMMUNITIES ===")
    comm_sizes = optimal_communities['community'].value_counts().sort_values(ascending=False)
    
    for i, (comm_id, size) in enumerate(comm_sizes.head(10).items()):
        anime_list = optimal_communities[optimal_communities['community'] == comm_id].index.tolist()
        print(f"\nCommunity {comm_id} ({size} anime):")
        # Convert to strings in case index values are integers
        anime_str_list = [str(anime) for anime in anime_list[:8]]
        print(", ".join(anime_str_list) + ("..." if len(anime_list) > 8 else ""))

# Interactive selection
print("\n" + "="*60)
print("CUSTOM RESOLUTION SELECTION")
print("\nBased on the analysis above, you can choose a specific resolution.")
print("Guidelines:")
print("- Resolution < 1.0: Fewer, larger communities")
print("- Resolution = 1.0: Default")
print("- Resolution > 1.0: More, smaller communities")
print("- Resolution > 5.0: Many small communities")

# Function to apply a specific resolution
def apply_resolution(sim_matrix, resolution, threshold=0.3):
    """Apply Louvain with a specific resolution"""
    
    adj_matrix = sim_matrix.copy()
    adj_matrix[adj_matrix < threshold] = 0
    G = nx.from_pandas_adjacency(adj_matrix)
    G.remove_edges_from(nx.selfloop_edges(G))
    
    communities = community_louvain.best_partition(G, weight='weight', resolution=resolution)
    community_df = pd.DataFrame.from_dict(communities, orient='index', columns=['community'])
    
    return community_df

# Example: Apply your chosen resolution
chosen_resolution = 2  # Adjust this based on the analysis
final_communities = apply_resolution(sim_matrix, chosen_resolution)
final_communities.to_csv(f'anime_communities_res_{chosen_resolution}.csv')